# 18 Multi_Stage_Training_RLHF_PPO_GRPO

## Problem

一个 DeepSeek-like 模型从最开始到最终可用，通常不是单次训练完成的。它会经历哪些阶段，RLHF、PPO、GRPO 分别在什么位置上发挥作用？

## Dependencies

建议先完成 `14_pretraining_data_and_objective.ipynb`、`15_sft_and_alignment_basics.ipynb`、`16_reward_model_and_rl_intro.ipynb`。


## Module_Goals

- 理解 DeepSeek-like 模型常见的多阶段训练顺序
- 理解 pretraining、continued training、SFT、reward model、RLHF 之间的衔接
- 理解 PPO 和 GRPO 的目标差异与使用场景
- 能把多轮训练中的数据、模型版本和反馈信号放到一张流程图里

## Scope_And_Approach

这个 notebook 先给出完整训练主线，再拆开说明每个阶段在解决什么问题。重点不是堆砌 RL 术语，而是把“模型最开始怎么来、后面怎么一轮一轮迭代”这条线讲清楚，并给出 PPO 与 GRPO 的最小计算直觉。


## Context_And_Motivation

从工程视角看，一个强模型的形成往往不是一步到位，而是按阶段塑形：

1. **Pretraining**：建立语言底座、知识共现和通用表示能力。
2. **Continued_Training / Domain_Adaptation**：往代码、数学、推理、长上下文等更关键能力上补强。
3. **SFT**：把模型推向更稳定的 instruction-following 分布。
4. **Reward_Model**：从 preference data 里学出可优化的偏好分数。
5. **RLHF / Preference_Optimization**：让 policy 进一步朝目标行为更新。

这条链路背后的逻辑很朴素：先把模型做成一个“会生成、会理解”的通用系统，再把它往“更适合特定用途”的方向不断收敛。


In [ ]:
training_pipeline = [
    '01_pretraining',
    '02_continued_training_or_domain_adaptation',
    '03_supervised_fine_tuning',
    '04_reward_model_training',
    '05_rlhf_or_preference_optimization',
]

stage_inputs = {
    'pretraining': 'web + code + docs + books + papers',
    'continued_training': 'math/code/reasoning/domain-focused corpora',
    'sft': 'instruction-response pairs',
    'reward_model': 'chosen-rejected preference pairs',
    'rlhf': 'policy rollouts + reward signal + reference policy',
}

stage_outputs = {
    'pretraining': 'general language backbone',
    'continued_training': 'domain-strengthened backbone',
    'sft': 'assistant-style checkpoint',
    'reward_model': 'preference scoring model',
    'rlhf': 'policy aligned to reward or relative preference',
}

print('training_pipeline =')
for step_id, step in enumerate(training_pipeline, start=1):
    print(f'{step_id}. {step}')

print('\nstage_inputs =')
for key, value in stage_inputs.items():
    print(f'- {key}: {value}')

print('\nstage_outputs =')
for key, value in stage_outputs.items():
    print(f'- {key}: {value}')


In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)

# 一个最小“多轮迭代”示意：不同阶段使用不同数据桶和目标
stage_token_budget = {
    'pretraining': 1000,
    'continued_training': 180,
    'sft': 25,
    'reward_model': 8,
    'rlhf': 12,
}

total = sum(stage_token_budget.values())
for stage, budget in stage_token_budget.items():
    print(stage, 'share =', round(budget / total, 4))


In [ ]:
# 一个最小版本流转示意
checkpoint_lineage = [
    'base_pretrained_ckpt',
    'continued_training_ckpt',
    'sft_ckpt',
    'rm_ckpt',
    'rlhf_or_grpo_ckpt',
]

for idx, ckpt in enumerate(checkpoint_lineage, start=1):
    print(f'v{idx}: {ckpt}')


In [ ]:
# PPO 的最小直觉：优势值 advantage 决定这次回答应该被推高还是压低
old_logprob = np.array([-1.20, -0.80, -1.50])
new_logprob = np.array([-1.00, -0.95, -1.10])
advantage = np.array([0.8, -0.4, 0.5])
clip_eps = 0.2

ratio = np.exp(new_logprob - old_logprob)
clipped_ratio = np.clip(ratio, 1 - clip_eps, 1 + clip_eps)
ppo_objective = np.minimum(ratio * advantage, clipped_ratio * advantage)

print('ratio =', ratio)
print('clipped_ratio =', clipped_ratio)
print('ppo_objective =', ppo_objective)


In [ ]:
# GRPO 的最小直觉：同一 prompt 采样一组回答，直接做组内相对比较
group_rewards = np.array([0.62, 0.55, 0.91, 0.48])
group_mean = group_rewards.mean()
group_std = group_rewards.std() + 1e-6
group_advantage = (group_rewards - group_mean) / group_std

print('group_rewards =', group_rewards)
print('group_mean =', group_mean)
print('group_advantage =', group_advantage)


## PPO_Vs_GRPO

可以把两者先粗略区分成下面这样：

- **PPO**：更接近经典 RLHF 路线，常见做法里会显式维护 old policy、reference policy、advantage，有较强的 update constraint 意识。
- **GRPO**：更强调同题多答、组内相对比较和更轻量的优势构造，适合把“哪一个更好”这件事直接放进组内比较里处理。

PPO 常见目标可以写成：

$$L^{PPO} = \mathbb{E}[\min(r_t A_t, \text{clip}(r_t, 1-\epsilon, 1+\epsilon) A_t)]$$

其中 `r_t` 是新旧策略概率比，`A_t` 是 advantage，clip 的作用是避免策略一步更新过猛。

GRPO 的关键直觉则是：

- 针对同一 prompt 采样出一组回答
- 对这组回答做相对奖励比较
- 利用组内标准化后的相对优势更新策略

## Trade_Offs_And_Risk_Points

- 只记阶段顺序，不理解每一轮训练解决的问题，就会看不清为什么需要多阶段迭代。
- 把 RLHF 理解成“模型从这里开始获得知识”。更准确地说，它主要是在已有能力之上修正行为分布。
- 忽略 continued training。很多模型在通用预训练后，还会用更集中的语料继续补强数学、代码或推理。
- 把 PPO 和 GRPO 当成两个换名算法，而不去理解它们处理 advantage 与更新组织方式的差异。
- 不追踪 checkpoint lineage，会很难说清一轮轮模型到底是怎么演化出来的。

## Checkpoints

- 用自己的话复述从 pretraining 到 RLHF 的完整顺序。
- 解释 continued training 为什么常常是实际项目里的关键阶段。
- 解释 PPO 中 clip 的作用是什么。
- 解释 GRPO 为什么强调 group-relative comparison。
- 说明为什么要区分 base pretrained checkpoint、SFT checkpoint 和 RLHF checkpoint。

## Summary

DeepSeek-like 模型的训练主线通常是分阶段推进的：先用大规模语料建立底座，再用更集中、更对任务有利的数据继续强化，之后通过 SFT、reward model 和 RLHF 把行为推向目标分布。PPO 更像经典的受约束策略更新路线，GRPO 更强调组内相对比较和更轻量的优势构造。真正的工程重点不只是某个公式，而是整条版本流转和数据流转是否清楚。

## Next_Module

下一步进入数据侧：先看这些训练数据最开始从哪里来，网页、代码、文档、问答和论文类语料是如何被采集出来的。
